In [110]:
%pip install -e ..
# Editable install of the local checkout (one level up from notebooks/), not upstream master:
# preview_pipeline is new, branch-local code that has not been merged/released yet.

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib
  Attempting uninstall: raidionicsrads
    Found existing installation: raidionicsrads 1.3.2
    Uninstalling raidionicsrads-1.3.2:
      Successfully uninstalled raidionicsrads-1.3.2
  Running setup.py develop for raidionicsrads
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [111]:
# Download the models required for the pipeline to resolve, plus the real
# patient-UnitTest2 test data (same patient as 03_run_postoperative_segmentation.ipynb).
# Real patient data is only needed to actually run the pipeline once for real, below,
# to get a genuine "already computed" baseline -- previewing a pipeline itself never
# reads any actual scan.
import os
import requests
import zipfile

patient_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Samples-RaidionicsRADSLib-UnitTest2.zip'
brain_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_Brain-v13.zip'
seq_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_SequenceClassifier-v13.zip'
rest_tumor_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_TumorCE_Postop-v13.zip'
cavity_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_Cavity-v13.zip'

test_dir = os.path.join(os.getcwd(), 'unit_tests_results_dir')
patient_dir = os.path.join(test_dir, 'patient')
models_dir = os.path.join(test_dir, 'models')
results_dir = os.path.join(test_dir, 'results')
os.makedirs(patient_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

archive_dl_dest = os.path.join(test_dir, 'inference_patient.zip')
if not os.path.exists(archive_dl_dest):
    response = requests.get(patient_url, stream=True)
    response.raise_for_status()
    with open(archive_dl_dest, "wb") as f:
        for chunk in response.iter_content(chunk_size=1048576):
            f.write(chunk)
with zipfile.ZipFile(archive_dl_dest, 'r') as zip_ref:
    zip_ref.extractall(patient_dir)

for archive_name, url in [('seq-model.zip', seq_model_url),
                          ('brain-model.zip', brain_model_url),
                          ('rest_tumor-model.zip', rest_tumor_model_url),
                          ('cavity-model.zip', cavity_model_url)]:
    archive_dl_dest = os.path.join(test_dir, archive_name)
    if not os.path.exists(archive_dl_dest):
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(archive_dl_dest, "wb") as f:
            for chunk in response.iter_content(chunk_size=1048576):
                f.write(chunk)
    with zipfile.ZipFile(archive_dl_dest, 'r') as zip_ref:
        zip_ref.extractall(models_dir)


In [113]:
# Prepare the pipeline -- same pipeline.json as for an actual run. The preview only
# resolves which task/model each step refers to, it never runs any of them.
import json

pipeline_dir = os.path.join(test_dir, 'pipelines')
os.makedirs(pipeline_dir, exist_ok=True)

pipeline_json = {}
step_index = 1
step_str = str(step_index)
pipeline_json[step_str] = {}
pipeline_json[step_str]["task"] = "Classification"
pipeline_json[step_str]["inputs"] = {}  # Empty input means running it on all existing data for the patient
pipeline_json[step_str]["target"] = ["MRSequence"]
pipeline_json[step_str]["model"] = "MRI_SequenceClassifier"
pipeline_json[step_str]["description"] = "Classification of the MRI sequence type for all input scans."

step_index = step_index + 1
step_str = str(step_index)
pipeline_json[step_str] = {}
pipeline_json[step_str]["task"] = 'Model selection'  # Will select the appropriate model for the provided set of MR scans
pipeline_json[step_str]["model"] = 'MRI_TumorCE_Postop'
pipeline_json[step_str]["timestamp"] = 1  # Timestamp 1 indicates early post-operative data, located in folder T1
pipeline_json[step_str]["format"] = "thresholding"
pipeline_json[step_str]["description"] = "Identifying the best rest tumor segmentation model for existing inputs"

step_index = step_index + 1
step_str = str(step_index)
pipeline_json[step_str] = {}
pipeline_json[step_str]["task"] = 'Model selection'
pipeline_json[step_str]["model"] = 'MRI_Cavity'
pipeline_json[step_str]["timestamp"] = 1
pipeline_json[step_str]["format"] = "thresholding"
pipeline_json[step_str]["description"] = "Identifying the best cavity segmentation model for existing inputs"

print(json.dumps(pipeline_json, indent=4))
with open(os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'), 'w', newline='\n') as outfile:
    json.dump(pipeline_json, outfile, indent=4, sort_keys=True)

{
    "1": {
        "task": "Classification",
        "inputs": {},
        "target": [
            "MRSequence"
        ],
        "model": "MRI_SequenceClassifier",
        "description": "Classification of the MRI sequence type for all input scans."
    },
    "2": {
        "task": "Model selection",
        "model": "MRI_TumorCE_Postop",
        "timestamp": 1,
        "format": "thresholding",
        "description": "Identifying the best rest tumor segmentation model for existing inputs"
    },
    "3": {
        "task": "Model selection",
        "model": "MRI_Cavity",
        "timestamp": 1,
        "format": "thresholding",
        "description": "Identifying the best cavity segmentation model for existing inputs"
    }
}


In [114]:
# Run the real pipeline once on the actual patient-UnitTest2 data, to get a genuine,
# reproducible "already computed" baseline for the reuse-check demonstration further
# below. This is the ONLY cell in this notebook that performs real computation (takes
# a few minutes on CPU) -- previewing itself never runs anything.
import configparser
from raidionicsrads.compute import run_rads

baseline_results_dir = os.path.join(results_dir, "output_postop_segmentation_baseline")
os.makedirs(baseline_results_dir, exist_ok=True)

baseline_config = configparser.ConfigParser()
baseline_config.add_section('Default')
baseline_config.set('Default', 'task', 'neuro_diagnosis')
baseline_config.set('Default', 'caller', '')
baseline_config.add_section('System')
baseline_config.set('System', 'gpu_id', '-1')
baseline_config.set('System', 'input_folder', os.path.join(patient_dir, 'patient-UnitTest2', 'inputs'))
baseline_config.set('System', 'output_folder', baseline_results_dir)
baseline_config.set('System', 'model_folder', models_dir)
baseline_config.set('System', 'pipeline_filename', os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'))
baseline_config.add_section('Runtime')
baseline_config.set('Runtime', 'reconstruction_method', 'thresholding')
baseline_config.set('Runtime', 'reconstruction_order', 'resample_first')
baseline_config.set('Runtime', 'use_stripped_data', 'True')
baseline_config.set('Runtime', 'use_registered_data', 'False')

baseline_config_filename = os.path.join(baseline_results_dir, 'rads_config.ini')
with open(baseline_config_filename, 'w') as outfile:
    baseline_config.write(outfile)

run_rads(baseline_config_filename)


INFO:root:Could not find a valid ANTs root repository at .
 
INFO:root:Starting pipeline for file: /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/pipelines/pipeline_postop_segmentation.json.
INFO:root:LOG: Pipeline setup - 3 steps.
INFO:root:LOG: Pipeline - Classification of the MRI sequence type for all input scans. - Begin (1/3)
INFO:root:Starting inference for folder: inputs, with model: /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/models/MRI_SequenceClassifier.
INFO:root:LOG: Classification - 3 steps.
INFO:root:LOG: Classification - Preprocessing - Begin (1/3)
INFO:root:LOG: Classification - Runtime: 0.7739498615264893 seconds.
INFO:root:LOG: Classification - Preprocessing - End (1/3)
INFO:root:LOG: Classification - Inference - Begin (2/3)
INFO:root:LOG: Classification - Runtime: 1.0194590091705322 seconds.
INFO:root:LOG: Classification - Inference


--------------------------------------------------------------------------------------
 Mapping parameters
--------------------------------------------------------------------------------------
 ANTSPATH is /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/raidionicsrads/ANTs/bin

 Dimensionality:           3
 Output name prefix:       /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/
 Fixed images:             /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/postop_t1gd_masked.nii.gz
 Moving images:            /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/postop_t1_masked.nii.gz
 Mask images

INFO:root:LOG: Pipeline - Registration from T1w to T1CE - Runtime: 42.01546812057495 seconds.
INFO:root:LOG: Pipeline - Registration from T1w to T1CE - End (4/19)
INFO:root:LOG: Pipeline - Apply registration from T1w to T1CE - Begin (5/19)


 Registration finished. The antsRegistration call was:
--------------------------------------------------------------------------------------
/Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/raidionicsrads/ANTs/bin/antsRegistration --verbose 1 --dimensionality 3 --float 0 --collapse-output-transforms 1 --output [ /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/,/Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/Warped.nii.gz,/Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/InverseWarped.nii.gz ] --interpolation Linear --use-histogram-matching 0 --winsorize-image-intensities [ 0.005,0.995 ] --init

INFO:root:LOG: Pipeline - Apply registration from T1w to T1CE - Runtime: 1.60630202293396 seconds.
INFO:root:LOG: Pipeline - Apply registration from T1w to T1CE - End (5/19)
INFO:root:LOG: Pipeline - Brain segmentation in FLAIR - Begin (6/19)
INFO:root:[SegmentationStep] Automatic segmentation skipped, results already existing.
INFO:root:LOG: Pipeline - Brain segmentation in FLAIR - Runtime: 0.002274036407470703 seconds.
INFO:root:LOG: Pipeline - Brain segmentation in FLAIR - End (6/19)
INFO:root:LOG: Pipeline - Registration from FLAIR to T1CE - Begin (7/19)
INFO:root:[RegistrationStep] Using cpp ANTs backend.
INFO:root:[RegistrationStep] ANTs root located in /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/raidionicsrads/ANTs.



--------------------------------------------------------------------------------------
 Mapping parameters
--------------------------------------------------------------------------------------
 ANTSPATH is /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/raidionicsrads/ANTs/bin

 Dimensionality:           3
 Output name prefix:       /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/
 Fixed images:             /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/postop_t1gd_masked.nii.gz
 Moving images:            /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/postop_flair_masked.nii.gz
 Mask ima

INFO:root:LOG: Pipeline - Registration from FLAIR to T1CE - Runtime: 36.77990984916687 seconds.
INFO:root:LOG: Pipeline - Registration from FLAIR to T1CE - End (7/19)
INFO:root:LOG: Pipeline - Apply registration from FLAIR to T1CE - Begin (8/19)


 Registration finished. The antsRegistration call was:
--------------------------------------------------------------------------------------
/Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/raidionicsrads/ANTs/bin/antsRegistration --verbose 1 --dimensionality 3 --float 0 --collapse-output-transforms 1 --output [ /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/,/Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/Warped.nii.gz,/Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/results/output_postop_segmentation_baseline/registration/InverseWarped.nii.gz ] --interpolation Linear --use-histogram-matching 0 --winsorize-image-intensities [ 0.005,0.995 ] --init

INFO:root:LOG: Pipeline - Apply registration from FLAIR to T1CE - Runtime: 1.5135979652404785 seconds.
INFO:root:LOG: Pipeline - Apply registration from FLAIR to T1CE - End (8/19)
INFO:root:LOG: Pipeline - Postoperative contrast enhancing tumor segmentation - Begin (9/19)
INFO:root:Starting inference for folder: inputs, with model: t1c_t1w_t2f_t1d.
INFO:root:LOG: Segmentation - 4 steps.
INFO:root:LOG: Segmentation - Preprocessing - Begin (1/4)
INFO:root:LOG: Segmentation - Runtime: 1.5693120956420898 seconds.
INFO:root:LOG: Segmentation - Preprocessing - End (1/4)
INFO:root:LOG: Segmentation - Inference - Begin (2/4)
100%|██████████| 2/2 [00:20<00:00, 10.44s/it]
INFO:root:LOG: Segmentation - Runtime: 21.068054914474487 seconds.
INFO:root:LOG: Segmentation - Inference - End (2/4)
INFO:root:LOG: Segmentation - Reconstruction - Begin (3/4)
INFO:root:LOG: Segmentation - Runtime: 0.268949031829834 seconds.
INFO:root:LOG: Segmentation - Reconstruction - End (3/4)
INFO:root:LOG: Segmentation 

In [115]:
# Declare the MR sequences known to be available, per timestamp, instead of pointing
# to real image files. T0 is the preoperative timestamp, T1 the early postoperative one.
# (This matches what patient-UnitTest2, used in 03_run_postoperative_segmentation.ipynb,
# actually contains -- T1-CE only preop, T1-CE/T1-w/FLAIR postop -- so the resolved
# pipeline below can be compared against a real run of the same data.)
sequences_declaration = {
    "T0": ["T1-CE"],
    "T1": ["T1-CE", "T1-w"]
}

# Prepare the configuration file -- input_folder is left empty since no real patient
# data is read for a preview.
import configparser
import logging

test_results_dir = os.path.join(results_dir, "output_preview_postop_segmentation")
os.makedirs(test_results_dir, exist_ok=True)

logging.basicConfig()
logging.getLogger().setLevel(logging.INFO)
rads_config = configparser.ConfigParser()
rads_config.add_section('Default')
rads_config.set('Default', 'task', 'neuro_diagnosis')
rads_config.set('Default', 'caller', '')
rads_config.add_section('System')
rads_config.set('System', 'gpu_id', "-1")
rads_config.set('System', 'input_folder', '')
rads_config.set('System', 'output_folder', test_results_dir)
rads_config.set('System', 'model_folder', models_dir)
rads_config.set('System', 'pipeline_filename', os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'))
rads_config.add_section('Runtime')
rads_config.set('Runtime', 'reconstruction_method', 'thresholding')
rads_config.set('Runtime', 'reconstruction_order', 'resample_first')
rads_config.set('Runtime', 'use_stripped_data', 'True')
rads_config.set('Runtime', 'use_registered_data', 'False')

rads_config_filename = os.path.join(test_results_dir, 'rads_config.ini')
with open(rads_config_filename, 'w') as outfile:
    rads_config.write(outfile)

In [116]:
# Build executed_pipeline.json from the declared sequences -- setup() only, nothing
# is actually computed.
from raidionicsrads.compute import preview_pipeline

executed_pipeline = preview_pipeline(config_filename=rads_config_filename,
                                     sequences_declaration=sequences_declaration)
print(json.dumps(executed_pipeline, indent=4))

INFO:root:Could not find a valid ANTs root repository at .
 
INFO:root:Starting pipeline preview for file: /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/pipelines/pipeline_postop_segmentation.json.
INFO:root:LOG: Pipeline setup - 3 steps.
INFO:root:LOG: Pipeline - Classification of the MRI sequence type for all input scans. - Begin (1/3)
INFO:root:Classification step not executed since marked as skippable.
INFO:root:LOG: Pipeline - Classification of the MRI sequence type for all input scans. - Runtime: 0.0006058216094970703 seconds.
INFO:root:LOG: Pipeline - Classification of the MRI sequence type for all input scans. - End (1/3)
INFO:root:LOG: Pipeline - Identifying the best rest tumor segmentation model for existing inputs - Begin (2/3)
INFO:root:LOG: Pipeline - Identifying the best rest tumor segmentation model for existing inputs - Runtime: 0.002395153045654297 seconds.
INFO:root:LOG: Pipeline - Identifying the bes

{
    "1": {
        "description": "Classification of the MRI sequence type for all input scans.",
        "inputs": {},
        "model": "MRI_SequenceClassifier",
        "target": [
            "MRSequence"
        ],
        "task": "Classification"
    },
    "2": {
        "task": "Segmentation",
        "inputs": {
            "0": {
                "timestamp": 1,
                "sequence": "T1-CE",
                "labels": null,
                "space": {
                    "timestamp": 1,
                    "sequence": "T1-CE"
                }
            }
        },
        "target": [
            "Brain"
        ],
        "model": "MRI_Brain",
        "format": "thresholding",
        "description": "Brain segmentation in T1CE",
        "inclusion": "required"
    },
    "3": {
        "task": "Segmentation",
        "inputs": {
            "0": {
                "timestamp": 1,
                "sequence": "T1-w",
                "labels": null,
                "spac

In [117]:
# Inspecting what actually landed on disk: only executed_pipeline.json plus empty
# per-timestamp scaffolding folders should be there -- no predictions, no working
# directories, since nothing was executed.
for root, dirs, files in os.walk(test_results_dir):
    level = root.replace(test_results_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or os.path.basename(test_results_dir)}/")
    for f in sorted(files):
        print(f"{indent}  {f}")

output_preview_postop_segmentation/
  executed_pipeline.json
  rads_config.ini
  registration/
  T1/
  T0/


In [107]:
# Sanity check: declaring fewer sequences (only T1-CE, no T1-w/FLAIR at T1) should
# make model selection resolve to the lighter "t1c"-only submodels instead of the
# "t1c_t1w_t2f_t1d" ones picked above -- demonstrates the preview reacts correctly to
# what is declared as available.
sparse_sequences_declaration = {
    "T0": ["T1-CE"],
    "T1": ["T1-CE"]
}

sparse_results_dir = os.path.join(results_dir, "output_preview_postop_segmentation_sparse")
os.makedirs(sparse_results_dir, exist_ok=True)
rads_config.set('System', 'output_folder', sparse_results_dir)
sparse_config_filename = os.path.join(sparse_results_dir, 'rads_config.ini')
with open(sparse_config_filename, 'w') as outfile:
    rads_config.write(outfile)

sparse_executed_pipeline = preview_pipeline(config_filename=sparse_config_filename,
                                            sequences_declaration=sparse_sequences_declaration)
for step in sparse_executed_pipeline.values():
    if step.get("task") == "Segmentation":
        print(step["model"], "--", step["description"])

INFO:root:Could not find a valid ANTs root repository at .
 
INFO:root:Starting pipeline preview for file: /Users/aurorajohansen/Documents/Sintef/raidionics/Raidionics/raidionics_rads_lib/notebooks/unit_tests_results_dir/pipelines/pipeline_postop_segmentation.json.
INFO:root:LOG: Pipeline setup - 3 steps.
INFO:root:LOG: Pipeline - Classification of the MRI sequence type for all input scans. - Begin (1/3)
INFO:root:Classification step not executed since marked as skippable.
INFO:root:LOG: Pipeline - Classification of the MRI sequence type for all input scans. - Runtime: 0.0009548664093017578 seconds.
INFO:root:LOG: Pipeline - Classification of the MRI sequence type for all input scans. - End (1/3)
INFO:root:LOG: Pipeline - Identifying the best rest tumor segmentation model for existing inputs - Begin (2/3)
INFO:root:LOG: Pipeline - Identifying the best rest tumor segmentation model for existing inputs - Runtime: 0.003831148147583008 seconds.
INFO:root:LOG: Pipeline - Identifying the bes

MRI_Brain -- Brain segmentation in T1CE
MRI_TumorCE_Postop/t1c -- Postoperative contrast enhancing tumor segmentation
MRI_Brain -- Brain segmentation in T1CE (T1)
MRI_Cavity/t1c -- Resection cavity segmentation


In [108]:
# would_step_be_skipped(): checks Segmentation(-refinement) and Registration/Apply
# registration steps against simple registries -- no fake Annotation/Registration
# objects (Registration.__init__ does shutil.copyfile on real files, so it can't be
# faked the way RadiologicalVolume was). Does not touch raidionicsrads or
# PatientParameters -- pure comparison against the fields already present in a
# resolved step from preview_pipeline(). Used by the real-data derivation below.

def would_step_be_skipped(step, already_computed_targets, already_computed_registrations):
    task = step.get("task")

    if task in ("Segmentation", "Segmentation refinement"):
        target = step.get("target")
        if not target:
            return None
        ts = step["inputs"]["0"]["timestamp"] if step.get("inputs") else step.get("timestamp")
        have = already_computed_targets.get(ts, set())
        return all(t in have for t in target)

    if task in ("Registration", "Apply registration"):
        moving, fixed = step["moving"], step["fixed"]
        key = (moving["sequence"], moving["timestamp"], fixed["sequence"], fixed["timestamp"])
        return key in already_computed_registrations

    return None  # Classification / Model selection / Reporting selection


status_label = {True: "ALREADY COMPUTED -- can be skipped",
                 False: "MISSING -- must run",
                 None: "(not relevant for this check)"}


In [118]:
# Derive already_computed_targets FOR REAL, by pointing the real, unmodified
# PatientParameters.__init_from_scratch() (NOT our declared_sequences path) at a folder
# combining real raw MR volumes with the real annotation output files from the baseline
# run above. This is what generate_surrogate_folder() (Raidionics GUI,
# utils/backend_logic.py) does in production: merge raw data + already-computed results
# into one input_folder.
#
# Note: had to rename the annotation files to the "<basename>_annotation-<Type>.nii.gz"
# convention (no model-name suffix) -- PatientStructure's reader (caller='raidionics')
# cannot parse the "<basename>_annotation-<Type>_<Model>.nii.gz" form that
# SegmentationStep itself sometimes writes (a real writer/reader mismatch, noted
# separately, not fixed here).
import shutil
from raidionicsrads.Utils.DataStructures.PatientStructure import PatientParameters
from raidionicsrads.Utils.configuration_parser import ResourcesConfiguration

patient_source_dir = os.path.join(patient_dir, 'patient-UnitTest2', 'inputs')

merged_dir = os.path.join(test_dir, 'merged_input_for_reuse_check')
if os.path.exists(merged_dir):
    shutil.rmtree(merged_dir)
os.makedirs(os.path.join(merged_dir, 'T0', 'raw'))
os.makedirs(os.path.join(merged_dir, 'T1', 'raw'))

# Real raw MR volumes
shutil.copy(os.path.join(patient_source_dir, 'T0', 'preop_t1gd.nii.gz'), os.path.join(merged_dir, 'T0', 'raw'))
for f in ['postop_flair.nii.gz', 'postop_t1.nii.gz', 'postop_t1gd.nii.gz']:
    shutil.copy(os.path.join(patient_source_dir, 'T1', f), os.path.join(merged_dir, 'T1', 'raw'))

# Real, already-computed annotation results -- renamed to the convention the reader
# actually supports (stripping the "_MRI_Cavity"/"_MRI_TumorCE_Postop" model suffix).
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1gd_annotation-Cavity_MRI_Cavity.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1gd_annotation-Cavity.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1gd_annotation-TumorCE_MRI_TumorCE_Postop.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1gd_annotation-TumorCE.nii.gz'))

# Real Brain segmentation results from the same baseline run -- these use a DIFFERENT
# naming convention on disk ("_label_Brain.nii.gz", written by the use_stripped_data
# auto-mask fallback in PatientStructure.py, not by SegmentationStep). Renamed to the
# same "_annotation-<Type>" convention for consistency in this test (use_stripped_data
# is set to False below, so this is testing genuine prior-run reuse, not the fallback).
shutil.copy(os.path.join(baseline_results_dir, 'T0', 'preop_t1gd_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T0', 'raw', 'preop_t1gd_annotation-Brain.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_flair_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_flair_annotation-Brain.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1_annotation-Brain.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1gd_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1gd_annotation-Brain.nii.gz'))

# caller must be 'raidionics' for the reader to look for a per-timestamp raw/ subfolder
# and to split filenames on 'annotation' rather than 'label'.
merged_config = configparser.ConfigParser()
merged_config.add_section('Default')
merged_config.set('Default', 'task', 'neuro_diagnosis')
merged_config.set('Default', 'caller', 'raidionics')
merged_config.add_section('System')
merged_config.set('System', 'gpu_id', '-1')
merged_config.set('System', 'input_folder', merged_dir)
merged_config.set('System', 'output_folder', os.path.join(results_dir, 'merged_input_scratch'))
merged_config.set('System', 'model_folder', models_dir)
merged_config.set('System', 'pipeline_filename', os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'))
merged_config.add_section('Runtime')
merged_config.set('Runtime', 'reconstruction_method', 'thresholding')
merged_config.set('Runtime', 'reconstruction_order', 'resample_first')
merged_config.set('Runtime', 'use_stripped_data', 'False')
merged_config.set('Runtime', 'use_registered_data', 'False')
os.makedirs(os.path.join(results_dir, 'merged_input_scratch'), exist_ok=True)
merged_config_filename = os.path.join(merged_dir, 'rads_config.ini')
with open(merged_config_filename, 'w') as outfile:
    merged_config.write(outfile)

ResourcesConfiguration.getInstance().set_environment(config_path=merged_config_filename)
patient_parameters = PatientParameters(id="Patient", patient_filepath=merged_dir)

print("Real annotations found by PatientParameters:")
already_computed_targets = {}
for anno_uid, anno in patient_parameters.annotation_volumes.items():
    vol = patient_parameters.get_radiological_volume(anno._radiological_volume_uid)
    ts = int(vol._timestamp_id[1:])  # "T1" -> 1
    already_computed_targets.setdefault(ts, set()).add(anno._annotation_type.name)
    print(f"  {anno_uid}: {anno.get_annotation_type_str()} @ T{ts} (volum: {vol.get_sequence_type_str()})")

print()

for k in sorted(executed_pipeline.keys(), key=int):
    step = executed_pipeline[k]
    # Hardkodet til en tom set() siden lesemekanismen for registrering ikke fungerer
    status = would_step_be_skipped(step, already_computed_targets, set())
    if status is not None:
        print(f"{k:>2} {step.get('task'):<24} target={str(step.get('target')):<12} -> {status_label[status]}")


INFO:root:Could not find a valid ANTs root repository at .
 


Real annotations found by PatientParameters:
  A3022_preop_t1gd: Brain @ T0 (volum: T1-CE)
  A15_postop_flair: Brain @ T1 (volum: FLAIR)
  A9244_postop_t1: Brain @ T1 (volum: T1-CE)
  A4820_postop_t1gd: Cavity @ T1 (volum: T1-CE)
  A9932_postop_t1gd: Contrast-enhancing tumor @ T1 (volum: T1-CE)
  A9303_postop_t1gd: Brain @ T1 (volum: T1-CE)

 2 Segmentation             target=['Brain']    -> ALREADY COMPUTED -- can be skipped
 3 Segmentation             target=['Brain']    -> ALREADY COMPUTED -- can be skipped
 4 Registration             target=None         -> MISSING -- must run
 5 Apply registration       target=None         -> MISSING -- must run
 6 Segmentation             target=['TumorCE']  -> ALREADY COMPUTED -- can be skipped
 8 Segmentation             target=['Brain']    -> ALREADY COMPUTED -- can be skipped
 9 Segmentation             target=['Brain']    -> ALREADY COMPUTED -- can be skipped
10 Registration             target=None         -> MISSING -- must run
11 Apply regi